# 6.6: RoPE: the score depends only on the offset


In [ ]:
import torch

def rope(x, pos, base=10000.0):
    """Rotate pairs of dimensions of x (..., d) by angles pos * omega_i."""
    d = x.size(-1)
    omega = base ** (-torch.arange(0, d, 2, dtype=x.dtype) / d)
    ang = pos * omega
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.empty_like(x)
    out[..., 0::2] = x1 * torch.cos(ang) - x2 * torch.sin(ang)
    out[..., 1::2] = x1 * torch.sin(ang) + x2 * torch.cos(ang)
    return out

torch.manual_seed(0)
q, k = torch.randn(64, dtype=torch.float64), torch.randn(64, dtype=torch.float64)
s1 = rope(q, 12.0) @ rope(k, 5.0)       # positions 12 and 5 (offset 7)
s2 = rope(q, 107.0) @ rope(k, 100.0)    # positions 107 and 100 (offset 7)
print(torch.allclose(s1, s2))           # True: the score depends only on the offset
